**Human Movement Intelligence | Biomechanical Load Estimation | Presented by: Reza Saadatyar (2026)**

**07 | Original FE Time Coordinates and Missing-Field Reconciliation**

![Original Essex FE time-coordinate mapping and missing-field audit](../../images/07_workflow.svg)

**Objective:** Establish whether original FE `timestep` coordinates uniquely identify corresponding original IMU samples, and reconcile all source-declared empty FE fields with their Python-decoded representations.

**Data source:** Hollinger et al. (2026), *Scientific Data*, [DOI 10.1038/s41597-026-07429-3](https://doi.org/10.1038/s41597-026-07429-3); original [Figshare version 13](https://doi.org/10.6084/m9.figshare.30225739.v13). The complete source contains 40 participants, 1,897 movement records and six activities.

**Evidence motivating the analysis:** The preceding full-source audit recovered 42 finite IMU channels for all 1,897 records and numeric `e1/e3` columns for 1,729. It observed 168 registry-declared empty FE fields, but its decoded empty-object test recognised only 24. It also found that 1,729 FE time vectors differed from the assumed fixed `IMU[::10]` indices. This notebook tests source-coordinate matching without that fixed-index assumption.

**Method:** Compare actual FE time values against all 101 original IMU time coordinates, require a unique, monotonic match for each FE position, preserve source record IDs, and separately describe the original Python classes returned for source-empty FE values.

**Interpretation:** Exact time-coordinate matching supports original sample-index linkage. It does not establish clinical validity, strain-unit conventions, model generalisation or complete model readiness. No interpolation or machine-learning fitting is performed.

**00. Colab Repository Setup**

In [1]:
# ======================== 00. Colab Repository Setup =========================

# -------------------------- 1. Repository Checkout ---------------------------
from pathlib import Path
repo_url = "https://github.com/RezaSaadatyar/human-movement-intelligence.git"
repo_path = Path("/content") / repo_url.split("/")[-1].removesuffix(".git")

%cd /content

if not repo_path.exists():
    !git clone -q {repo_url}


# ---------------------------- 2. Open Repository -----------------------------
%cd {repo_path}

/content
/content/human-movement-intelligence


**00A. Original Research Source Availability**

A reused Colab runtime may retain an earlier repository checkout. This check refreshes the source tree **only if** the study's temporal-mapping module is absent. Existing local work is never force-overwritten.

In [2]:
# ==================== 00A. Original Study Source Check =====================

module_path = (
    repo_path / "biomechanical_load_estimation" / "code"
    / "src" / "essex_temporal_mapping.py"
)
if not module_path.is_file():
    !git -C {repo_path} pull --ff-only -q
if not module_path.is_file():
    raise FileNotFoundError(
        "Current Colab clone lacks essex_temporal_mapping.py. "
        "Start a fresh runtime and open the latest GitHub notebook."
    )
print("Study temporal-mapping source: available")

error: The following untracked working tree files would be overwritten by merge:
	biomechanical_load_estimation/output/06_activity_numeric_coverage.csv
	biomechanical_load_estimation/output/06_imu_channel_contract.csv
	biomechanical_load_estimation/output/06_numeric_alignment_diagnostics.png
	biomechanical_load_estimation/output/06_numeric_timebase_quality_gate.json
	biomechanical_load_estimation/output/06_original_numeric_examples.png
	biomechanical_load_estimation/output/06_original_numeric_record_audit.csv
	biomechanical_load_estimation/output/06_participant_numeric_coverage.csv
	biomechanical_load_estimation/output/06_reader_errors.csv
	biomechanical_load_estimation/output/06_time_alignment_status.csv
	biomechanical_load_estimation/output/06_timebase_metadata.csv
Please move or remove them before you merge.
Aborting


FileNotFoundError: Current Colab clone lacks essex_temporal_mapping.py. Start a fresh runtime and open the latest GitHub notebook.

**01. Imports and Environment**

In [ ]:
# ======================== 01. Imports and Environment ========================

from python_modules.core.colab_setup import setup_colab
root = setup_colab("biomechanical_load_estimation")

import json
from pathlib import Path
from importlib.metadata import version

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from core.plot_style import plot_style, save_figure
from core.output_download import download_outputs_from_colab
from essex_figshare import (
    ARTICLE_ID, ARTICLE_VERSION, fetch_version_manifest,
    find_published_file, ensure_source_file,
    inspect_matlab_archive, extract_matlab_files,
)
from essex_numeric_recovery import check_record_registry
from essex_temporal_mapping import (
    audit_temporal_mapping, summarise_temporal_evidence,
)

plot_style(font_family="DejaVu Sans", title_size=12.5,
    title_weight="normal", label_size=11.5, label_weight="normal",
    tick_size=11, legend_size=10, figure_dpi=100,
    figure_size=(9, 4))
study_dir = root / "biomechanical_load_estimation"
raw_dir = root / "data" / "raw" / "essex_femoral_neck_loading"
mat_dir = raw_dir / "matlab_records"
output_dir = study_dir / "output"
output_dir.mkdir(parents=True, exist_ok=True)
print("Study dataset:", ARTICLE_ID, ARTICLE_VERSION)

**02. Python MATLAB Object Reader**

The original MATLAB tables are read using `mat-io==1.0.1` in a Colab Python 3.11–3.13 environment. The pinned binary wheel is installed without automatically changing existing scientific packages. MATLAB Desktop is not required for the source-coordinate analysis.

In [ ]:
# ================= 02. Load the Pinned Python MATLAB Reader ==================

%pip -q install --only-binary=:all: --no-deps "mat-io==1.0.1"

from matio import load_from_mat

assert version("mat-io") == "1.0.1"
print("Original MATLAB reader:", version("mat-io"))

**03. Official Source Integrity**

Verify the original Figshare version 13 archive and original MAT participant files against the existing source SHA256. The numerical audit uses the complete source, rather than a reduced derivative dataset.

In [ ]:
# =============== 03. Verify the Original Essex Source Archive ================

source_gate = json.loads(
    (output_dir / "02_record_provenance_and_quality_gate.json")
    .read_text(encoding="utf-8")
)
prior_gate = json.loads(
    (output_dir / "06_numeric_timebase_quality_gate.json")
    .read_text(encoding="utf-8")
)
assert source_gate["source_article_id"] == ARTICLE_ID
assert source_gate["version"] == ARTICLE_VERSION
assert prior_gate["source_integrity"] == "PASSED"
assert prior_gate["source_archive_sha256"] == source_gate["archive_sha256"]
_, published = fetch_version_manifest()
source_item = find_published_file(published, "MATLAB.zip")
source_zip, digest = ensure_source_file(source_item, raw_dir)
assert digest["sha256"] == source_gate["archive_sha256"]
inventory = inspect_matlab_archive(
    source_zip, expected_mat_files=40
)
files = extract_matlab_files(source_zip, mat_dir, inventory)
assert len(files) == 40
print("Original source SHA256:", digest["sha256"])

**04. Original Record Registry and Missingness Contract**

The source-defined `e1e3` representation is the reference for empty FE records. An empty field in the original registry is kept separate from the reader's Python class and shape; inconsistent representations are investigated, not silently converted into strain values.

In [ ]:
# =========== 04. Freeze Source Trial Identities and FE Missingness ===========

registry = pd.read_csv(
    output_dir / "02_original_record_registry.csv",
    dtype={
        "subject_id": "string", "source_file": "string",
        "local_record_key": "string", "task": "string",
        "updated_trial_id": "string", "fe_representation": "string",
    },
)
governance = check_record_registry(registry, source_gate)
assert governance["records"] == 1897
assert governance["participants"] == 40
assert registry["fe_representation"].eq("empty").sum() == 168
assert {p.stem.lower() for p in files} == set(registry["subject_id"])
print("Original source records:", len(registry))
print("Registry-declared empty FE fields:", 168)

**05. Source-Time Matching and FE Empty Representations**

For nonempty FE records, match every original FE time coordinate against the full IMU `timestep` vector. Each FE coordinate must identify exactly one IMU sample and preserve temporal order. This tests the original values directly, rather than assuming an evenly spaced positional slice.

For empty FE records, retain the source-defined missingness label and record the decoded object's class, array shape and, where applicable, table schema and numeric-field completeness. Apparent nonempty reader representations are investigated rather than automatically treated as strain targets.

In [ ]:
# ======== 05. Audit All Original Time Coordinates and FE Empty Fields ========

mapping, fe_empty, reader_errors, source_example = (
    audit_temporal_mapping(load_from_mat, mat_dir, registry)
)
assert len(mapping) == 1897
assert mapping["local_record_key"].is_unique
assert len(fe_empty) == 168

mapping.to_csv(
    output_dir / "07_original_time_coordinate_mapping.csv", index=False
)
fe_empty.to_csv(
    output_dir / "07_source_fe_empty_representations.csv", index=False
)
reader_errors.to_csv(
    output_dir / "07_source_reader_errors.csv", index=False
)
print("All original movement records:", len(mapping))
print("Original FE-empty representations inspected:", len(fe_empty))
print("MAT reader errors:", len(reader_errors))
display(mapping[[
    "source_file", "task", "fe_channel_status",
    "mapping_status", "imu_position_indices",
]].head(20))

**06. FE Missingness and Reader-Representation Reconciliation**

The original registry identifies **168 FE-empty records**. The prior full-source audit recognised 24 decoded values as directly empty and left another 144 with different representations.

This section classifies all 168 records by Python type, dimensions and source-column structure. Where the reader returns a nonempty table, numerical-field completeness is described without overriding the original missingness label or creating an FE target.

In [ ]:
# ================ 06. Review Source-Declared FE Empty Classes ================

missingness = (
    fe_empty.groupby([
        "python_class", "python_shape",
        "empty_representation_status", "column_names",
        "fe_numeric_columns_finite",
    ], dropna=False)
    .size().rename("source_records").reset_index()
)
missingness.to_csv(
    output_dir / "07_fe_missingness_types.csv", index=False
)
display(missingness)
print("Total source-declared FE empties:",
      missingness["source_records"].sum())

**07. Direct IMU Sample-Index Mapping**

For nonempty FE fields, summarise exact and tolerance-based coordinate matches, index step patterns and unresolved cases. The saved per-record `imu_position_indices` preserve the actual source mapping for subsequent scientific review. No interpolation is needed when a unique observed IMU time coordinate is available.

In [ ]:
# ============== 07. Review Source Coordinate Mapping Signatures ==============

mapping_signatures = (
    mapping.groupby([
        "mapping_status", "consecutive_index_steps",
    ], dropna=False).size().rename("source_records").reset_index()
)
mapping_signatures.to_csv(
    output_dir / "07_time_mapping_signatures.csv", index=False
)
display(mapping_signatures.head(30))
print("Original mapping status distribution:")
print(mapping["mapping_status"].value_counts())

**08. Activity and Participant Source Coverage**

Inspect whether FE missingness, source-coordinate mappings and unresolved structures are concentrated in particular activities or participants. Count source movement records, rather than treating each of the 11 FE sample positions as an independent trial.

In [ ]:
# =============== 08. Participant and Activity Mapping Coverage ===============

def group_coverage(source, by):
    return source.groupby(by).agg(
        original_records=("local_record_key", "size"),
        source_empty_fe=("originally_empty_fe", "sum"),
        coordinate_mappings=(
            "mapping_status",
            lambda s: s.isin([
                "EXACT_TIME_COORDINATE_MAPPING",
                "TOLERANT_TIME_COORDINATE_MAPPING",
            ]).sum(),
        ),
        unresolved_time=(
            "mapping_status",
            lambda s: s.eq("TIME_VALUES_NOT_AVAILABLE").sum(),
        ),
    ).reset_index()

participant_report = group_coverage(mapping, "subject_id")
activity_report = group_coverage(mapping, "task")
participant_report.to_csv(
    output_dir / "07_participant_coordinate_coverage.csv", index=False
)
activity_report.to_csv(
    output_dir / "07_activity_coordinate_coverage.csv", index=False
)
display(activity_report)
print("Participants reported:", len(participant_report))

**09. Authentic Source-Coordinate Figures**

The first figure compares available source mappings and FE missingness by activity, alongside the recorded temporal-index patterns. The second figure shows the actual original IMU and FE time coordinates from one mapped record, if one is available.

These are visualisations of source time coordinates, not predictions or strain reconstruction.

In [ ]:
# =========== 09. Visualise Original FE to IMU Time-Index Evidence ============

fig, axes = plt.subplots(
    1, 3, figsize=(18, 5.1),
    gridspec_kw={"width_ratios": [1.6, 1.1, 1.1]},
    constrained_layout=True,
)
pos = np.arange(len(activity_report))
axes[0].bar(
    pos - 0.17, activity_report["coordinate_mappings"],
    width=0.33, label="Mapped FE time vectors",
)
axes[0].bar(
    pos + 0.17, activity_report["source_empty_fe"],
    width=0.33, label="Source-empty FE",
)
axes[0].set_xticks(pos)
axes[0].set_xticklabels(
    activity_report["task"].str.replace(
        "Hopping", " Hopping", regex=False
    ), rotation=26, ha="right",
)
axes[0].set_title("Original FE coordinate coverage")
axes[0].set_ylabel("Movement records")
axes[0].legend(frameon=False)
axes[0].grid(axis="y", alpha=0.2)

state_counts = mapping["mapping_status"].value_counts().sort_values()
axes[1].barh(
    np.arange(len(state_counts)), state_counts.values, height=0.32
)
axes[1].set_yticks(np.arange(len(state_counts)))
axes[1].set_yticklabels(
    [s.replace("_", " ").title() for s in state_counts.index]
)
axes[1].set_title("Original mapping classifications")
axes[1].set_xlabel("Movement records")
axes[1].grid(axis="x", alpha=0.2)

class_counts = (
    fe_empty["empty_representation_status"].value_counts()
)
axes[2].barh(
    np.arange(len(class_counts)), class_counts.values, height=0.32
)
axes[2].set_yticks(np.arange(len(class_counts)))
axes[2].set_yticklabels(
    [s.replace("_", " ").title() for s in class_counts.index]
)
axes[2].set_title("Source-empty FE representations")
axes[2].set_xlabel("Original source-empty records")
axes[2].grid(axis="x", alpha=0.2)
save_figure(fig, output_dir / "07_temporal_mapping_diagnostics.png")
plt.show()

fig2, axs = plt.subplots(1, 2, figsize=(13, 4.5),
                         constrained_layout=True)
if source_example is not None:
    actual_imu = source_example["imu_time"]
    actual_fe = source_example["fe_time"]
    mapped = np.asarray(source_example["imu_indices"])
    axs[0].plot(
        np.arange(len(actual_imu)), actual_imu,
        linewidth=1.2, label="Original IMU time"
    )
    axs[0].scatter(
        mapped, actual_fe, s=24, label="Original FE time", zorder=3
    )
    axs[0].set_title("Native FE coordinates on original IMU positions")
    axs[0].set_xlabel("Original IMU sample index (zero-based)")
    axs[0].set_ylabel("Original time coordinate")
    axs[0].legend(frameon=False)
    axs[1].plot(
        np.arange(len(actual_fe)), mapped, "o-",
        linewidth=1.2
    )
    axs[1].set_title("Source FE-to-IMU sample mapping")
    axs[1].set_xlabel("Original FE sample index")
    axs[1].set_ylabel("Matched IMU sample index")
else:
    for axis in axs:
        axis.text(0.5, 0.5, "No unique original time mapping",
                  ha="center", va="center",
                  transform=axis.transAxes)
        axis.set_xticks([])
        axis.set_yticks([])
save_figure(fig2, output_dir / "07_original_coordinate_example.png")
plt.show()

**10. Quality Gate and Research Outputs**

The record-level Quality Gate reports actual source-coordinate mapping coverage and the classification of all 168 source-empty FE objects. It distinguishes coordinate consistency from independent verification of physical units and target semantics.

Ten `07_*` evidence files contain the record-level mappings, FE empty-class audit, participant/activity reports, figures and Quality Gate JSON. In Google Colab, the final cell packages the research outputs into a downloadable ZIP.

In [ ]:
# ======== 10. Record Scientific Temporal Mapping Quality and Evidence ========

summary = summarise_temporal_evidence(
    mapping, fe_empty, reader_errors
)
final_evidence = {
    "source_article": ARTICLE_ID,
    "source_version": ARTICLE_VERSION,
    "source_archive_sha256": digest["sha256"],
    "reader": "mat-io",
    "reader_version": version("mat-io"),
    "source_integrity": "PASSED",
    "prior_full_source_audit": {
        "source_records": 1897,
        "imu_42_channel_records": 1897,
        "fe_e1_e3_records": 1729,
        "registry_declared_fe_empty": 168,
    },
    "source_temporal_mapping_audit": summary,
    "full_numeric_data_exported": False,
    "physical_units_verified": False,
    "model_ready": False,
}
(output_dir / "07_temporal_mapping_quality_gate.json").write_text(
    json.dumps(final_evidence, indent=2) + "\n",
    encoding="utf-8"
)
display(pd.DataFrame([summary]))
print("Source time-coordinate matching completed.")
print("Model readiness: FALSE")

required = [
    "07_original_time_coordinate_mapping.csv",
    "07_source_fe_empty_representations.csv",
    "07_source_reader_errors.csv",
    "07_fe_missingness_types.csv",
    "07_time_mapping_signatures.csv",
    "07_participant_coordinate_coverage.csv",
    "07_activity_coordinate_coverage.csv",
    "07_temporal_mapping_diagnostics.png",
    "07_original_coordinate_example.png",
    "07_temporal_mapping_quality_gate.json",
]
missing = [x for x in required if not (output_dir / x).is_file()]
if missing:
    raise FileNotFoundError(
        "Incomplete temporal mapping evidence: " + ", ".join(missing)
    )
print("Research evidence files:", len(required))
download_outputs_from_colab(
    folder="biomechanical_load_estimation/output", stage="07"
)